In [1]:
import torch
import clip
from PIL import Image

class PromptImageCLIPScorer:
    def __init__(self, model_name="clip-vit-large-patch14/ViT-L-14.pt", device=None):
        self.device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        self.model, self.preprocess = clip.load(model_name, device=self.device)
        self.random_features = self._generate_random_features()

    @torch.no_grad()
    def _generate_random_features(self):
        random_prompts = ["", "sks", "asfasdf"]
        tokens = clip.tokenize(random_prompts).to(self.device)
        features = self.model.encode_text(tokens)
        features = features / features.norm(dim=-1, keepdim=True)
        return features

    @torch.no_grad()
    def score_prompt_image(self, prompt: str, image: Image.Image) -> dict:
        image_input = self.preprocess(image).unsqueeze(0).to(self.device)
        text_input = clip.tokenize([prompt]).to(self.device)

        image_features = self.model.encode_image(image_input)
        text_features = self.model.encode_text(text_input)

        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        similarity = torch.sum(image_features * text_features, dim=-1).item()
        sim_image_random = torch.cosine_similarity(image_features, self.random_features).mean().item()
        sim_text_random = torch.cosine_similarity(text_features, self.random_features).mean().item()

        return {
            "image_text_similarity": round(similarity, 4),
            "image_random_similarity": round(sim_image_random, 4),
            "text_random_similarity": round(sim_text_random, 4)
        }

scorer = PromptImageCLIPScorer()


In [10]:
from PIL import Image
# Load a test image
image = Image.open(r"target_images\sport car.png")  # Replace with actual image path

# Define a test prompt
prompt = "A sleek red sports car driving along a winding mountain road, surrounded by towering cliffs and pine trees, under dramatic sky and cinematic lighting"

# Compute similarity score
score = scorer.score_prompt_image(prompt, image)

print(score)


{'image_text_similarity': 0.3013, 'image_random_similarity': 0.1177, 'text_random_similarity': 0.0355}
